# BFE-Net: Bio-inspired Feature Enhancement Network

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: Retina and LGN-inspired feature extraction  
**Deep Learning Enhancement**: Feature enhancement modules  
**Improvement Area**: Enhanced edge saliency

Inspired by retinal ganglion cells and Lateral Geniculate Nucleus (LGN).

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('..') / 'bio DL' / 'outputs' / 'BFE-Net'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## BFE-Net with Retinal-LGN Inspiration

In [ ]:
class RetinalGanglionLayer(nn.Module):
    """Retinal ganglion cells (center-surround)"""
    def __init__(self, channels):
        super().__init__()
        self.on_center = nn.Conv2d(channels, channels, 3, padding=1)
        self.off_surround = nn.Conv2d(channels, channels, 5, padding=2)
    def forward(self, x):
        on = self.on_center(x)
        off = self.off_surround(x)
        return F.relu(on - off)  # On-center, off-surround

class LGNLayer(nn.Module):
    """LGN-inspired feature enhancement"""
    def __init__(self, channels):
        super().__init__()
        self.enhance = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.relu(x + self.enhance(x))  # Residual enhancement

class BFENet(nn.Module):
    def __init__(self):
        super().__init__()
        self.retina = RetinalGanglionLayer(3)
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.lgn1 = LGNLayer(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.lgn2 = LGNLayer(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.edge = nn.Conv2d(256, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        # Retinal processing
        x = self.retina(x)
        # LGN-enhanced pathway
        x = F.relu(self.conv1(x))
        x = self.lgn1(x)
        x = F.relu(self.conv2(F.max_pool2d(x, 2)))
        x = self.lgn2(x)
        x = F.relu(self.conv3(F.max_pool2d(x, 2)))
        return torch.sigmoid(F.interpolate(self.edge(x), (h, w), mode='bilinear'))

model = BFENet().to(DEVICE).eval()
print(f"✓ BFE-Net: {sum(p.numel() for p in model.parameters()):,} params")

In [ ]:
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt = cv2.imread(str(self.gt_dir / img_path.name.replace('.jpg', '.png')), 0)
        gt = gt.astype(np.float32) / 255.0 if gt is not None else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

loader = DataLoader(EdgeDataset(Path('..') / 'datasets' / 'HED_Small', 'test'), batch_size=1)
preds, gts = [], []
with torch.no_grad():
    for imgs, gt, _ in tqdm(loader):
        preds.extend([model(imgs.to(DEVICE))[i,0].cpu().numpy() for i in range(imgs.shape[0])])
        gts.extend([gt[i].cpu().numpy() for i in range(gt.shape[0])])

def eval_metrics(preds, labels):
    t, ois, ap, al = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p = cv2.GaussianBlur(p, (3,3), 0).flatten()
        ap.append(p); al.append(l)
        ois.append(max([2*np.sum((p>=th)*l)/(2*np.sum((p>=th)*l)+np.sum((p>=th)*(1-l))+np.sum((p<th)*l)+1e-8) for th in t]))
    fp, fl = np.concatenate(ap), np.concatenate(al)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': ods[0], 'ODS_thresh': ods[1], 'OIS': np.mean(ois), 'AP': average_precision_score(fl, fp) if np.sum(fl)>0 else 0}

m = eval_metrics(preds, gts)
print(f"\nBFE-Net: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

import json
with open(OUTPUT_DIR / 'bfenet_metrics.json', 'w') as f:
    json.dump({'model': 'BFE-Net', 'bio': 'Retina + LGN', 'improvement': 'Edge saliency', 'metrics': m}, f, indent=2)
print("✅ BFE-Net complete!")